# HyDE: reproduction on SciFact and ArguAna

The claim we test (Gao et al., ACL 2023, Table 2): searching with the embedding of LLM-written hypothetical documents beats searching with the query embedding, with the same unsupervised encoder (Contriever) and no training.

Paper, nDCG@10: SciFact 64.9 with Contriever and 69.1 with HyDE; ArguAna 37.9 and 46.6.

Same as the paper: `facebook/contriever`, mean pooling, dot product, full documents, the paper's prompts, N = 8, temperature 0.7, at most 512 tokens, the query vector included in the average.
Different: the generator. InstructGPT (`text-davinci-003`) is retired; we use Qwen2.5-3B-Instruct, 8-bit, on a laptop (`generate.py`).

Run from the repo root after `generate.py`, or with the committed `generations/`. Scores come from `testbed.score_run` (`pytrec_eval`; on ArguAna the query's own document is removed).

In [ ]:
import sys
sys.path += ["../..", "."]
from sentence_transformers import SentenceTransformer
import testbed, hyde

model = SentenceTransformer("facebook/contriever", device=testbed.device())   # mean pooling
model.max_seq_length = 512

## Plain Contriever, then HyDE

For each dataset: the plain Contriever score (the check against the paper), then HyDE for N = 1, 2, 4, 8 hypothetical documents, with and without the query vector in the average. Every HyDE line is compared with plain Contriever on the same queries: mean difference, bootstrap 95 % CI, paired t-test.

In [ ]:
for name in ["scifact", "arguana"]:
    corpus, queries, qrels = testbed.load(name)
    ids, qids = list(corpus), list(queries)
    docs = [(d["title"] + " " + d["text"]).strip() for d in corpus.values()]    # full text, no 200-word cut
    D = model.encode(docs, batch_size=32, convert_to_numpy=True)
    Q = model.encode(list(queries.values()), batch_size=32, convert_to_numpy=True)
    E = hyde.embed_generations(model, queries, hyde.load_generations(name))
    _, base, base_r = testbed.score_run(hyde.search(Q, D, qids, ids), name)
    print(f"{name} Contriever            nDCG@10 {base.mean():.3f}  R@100 {base_r.mean():.3f}")
    for n in (1, 2, 4, 8):
        for label, q in (("with query", Q), ("docs only ", None)):
            _, nd, rc = testbed.score_run(hyde.search(hyde.hyde_vectors(E, q, n), D, qids, ids), name)
            c = testbed.compare(nd, base)
            print(f"{name} HyDE N={n} {label}  nDCG@10 {nd.mean():.3f}  R@100 {rc.mean():.3f}  "
                  f"delta {c['delta']:+.3f} {c['ci95']} p={c['p']:.3g}")